# Bài 2 — Xử lý luồng với PySpark Structured Streaming
`bai2_streaming.ipynb`

---

Xây dựng **lớp tốc độ (speed layer)**: đọc luồng giao dịch từ Kafka, tính tổng giá trị theo cửa sổ thời gian, gắn cờ giao dịch bất thường.

**Mục tiêu:**
- Kết nối PySpark Structured Streaming với Kafka
- Áp dụng windowed aggregation
- Phát hiện giao dịch bất thường theo thời gian thực

> ⚠️ Producer từ Bài 1 phải đang chạy song song.

## Phần A — Khởi tạo và đọc stream từ Kafka

In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

KAFKA_BOOTSTRAP = os.environ.get("KAFKA_BOOTSTRAP_SERVERS", "kafka:9093")
TOPIC_NAME      = "transactions"
DATA_OUTPUT     = os.environ.get("DATA_OUTPUT_PATH", "/data/output")

spark = SparkSession.builder.appName("Bai2_Streaming").getOrCreate()
spark.sparkContext.setLogLevel("WARN")
print(f"Spark master: {spark.sparkContext.master}")

In [ ]:
TRANSACTION_SCHEMA = StructType([
    StructField("transaction_id", StringType()),
    StructField("user_id",        StringType()),
    StructField("product_id",     StringType()),
    StructField("amount",         DoubleType()),
    StructField("category",       StringType()),
    StructField("timestamp",      StringType()),
    StructField("is_fraud",       BooleanType()),
])

In [ ]:
def read_kafka_stream():
    """
    Đọc streaming DataFrame từ Kafka, parse JSON, thêm cột event_time.
    Trả về DataFrame với schema TRANSACTION_SCHEMA + event_time (TimestampType).
    """
    # TODO: đọc từ Kafka (startingOffsets="latest")
    # TODO: parse cột value sang JSON theo TRANSACTION_SCHEMA
    # TODO: thêm cột event_time = cast timestamp sang TimestampType
    pass


stream_df = read_kafka_stream()
stream_df.printSchema()
assert "event_time" in [f.name for f in stream_df.schema.fields]
assert stream_df.isStreaming
print("✓ read_kafka_stream() hợp lệ")

## Phần B — Windowed aggregation

Tính tổng theo cửa sổ trượt: rộng **1 phút**, bước **30 giây**, nhóm theo `category`.
Kết quả cần có: `window`, `category`, `total_amount`, `txn_count`.

In [ ]:
def windowed_aggregation(df):
    # TODO: groupBy window + category, agg sum và count
    pass


windowed_df = windowed_aggregation(stream_df)
windowed_df.printSchema()
assert all(c in [f.name for f in windowed_df.schema.fields]
           for c in ["window","category","total_amount","txn_count"])
print("✓ windowed_aggregation() hợp lệ")

## Phần C — Phát hiện giao dịch bất thường

| Điều kiện | `anomaly_reason` |
|---|---|
| `amount > 3000` | `"high_value"` |
| `amount > 1000` VÀ `category == "food"` | `"unusual_category"` |
| Bình thường | `None` |

In [ ]:
def flag_anomalies(df):
    # TODO: thêm cột anomaly_reason (dùng F.when) và anomaly_flag (isNotNull)
    pass


# Kiểm tra với dữ liệu tĩnh
test_data = [
    ("txn-1","u1","p1",3500.0,"electronics","2024-01-01T10:00:00",False),
    ("txn-2","u2","p2",1200.0,"food",        "2024-01-01T10:01:00",False),
    ("txn-3","u3","p3", 100.0,"books",       "2024-01-01T10:02:00",False),
]
test_cols = ["transaction_id","user_id","product_id","amount","category","timestamp","is_fraud"]
test_df = spark.createDataFrame(test_data, test_cols)

result_df = flag_anomalies(test_df)
result_df.select("transaction_id","amount","category","anomaly_flag","anomaly_reason").show()

rows = {r.transaction_id: r for r in result_df.collect()}
assert rows["txn-1"].anomaly_reason == "high_value"
assert rows["txn-2"].anomaly_reason == "unusual_category"
assert rows["txn-3"].anomaly_flag   == False
print("✓ flag_anomalies() hợp lệ")

## Phần D — Chạy và quan sát

> ⚠️ Nếu sau 30 giây không thấy output, kiểm tra producer đang chạy và `KAFKA_BOOTSTRAP=kafka:9093`.

In [ ]:
CKPT_WINDOWED  = f"{DATA_OUTPUT}/checkpoints/windowed"
CKPT_ANOMALIES = f"{DATA_OUTPUT}/checkpoints/anomalies"
anomaly_df = flag_anomalies(stream_df)

query_windowed = windowed_df.writeStream     .outputMode("update").format("console")     .option("truncate", False)     .option("checkpointLocation", CKPT_WINDOWED)     .trigger(processingTime="30 seconds").start()

print("✓ Query khởi động — chờ ~30 giây")

In [ ]:
# TODO: thêm query thứ hai ghi anomaly_flag=True ra Parquet
# - path: f"{DATA_OUTPUT}/anomalies"
# - outputMode: "append"
# - checkpointLocation: CKPT_ANOMALIES
query_anomalies = ...  # TODO


In [ ]:
import time
print("Chạy 120 giây...")
time.sleep(120)
query_windowed.stop()
query_anomalies.stop()
print("✓ Đã dừng")

**Câu 1:** Trong 2 phút chạy có bao nhiêu cửa sổ được tạo? Danh mục nào có `total_amount` cao nhất?

*(Trả lời tại đây)*

**Câu 2:** `outputMode("update")` in ra những hàng nào mỗi trigger? Tại sao không dùng `"complete"`?

*(Trả lời tại đây)*

## Kiểm tra đầu ra

In [ ]:
import os
print("=" * 45)
print("KIỂM TRA ĐẦU RA BÀI 2")
print("=" * 45)

checks = [
    ("read_kafka_stream() trả về streaming DF",   stream_df.isStreaming),
    ("Có cột event_time", "event_time" in [f.name for f in stream_df.schema.fields]),
    ("windowed_aggregation() đủ 4 cột",
        all(c in [f.name for f in windowed_aggregation(stream_df).schema.fields]
            for c in ["window","category","total_amount","txn_count"])),
]
try:
    r = {row.transaction_id: row for row in flag_anomalies(test_df).collect()}
    checks.append(("flag_anomalies() đúng logic",
        r["txn-1"].anomaly_reason=="high_value" and
        r["txn-2"].anomaly_reason=="unusual_category" and
        r["txn-3"].anomaly_flag==False))
except: checks.append(("flag_anomalies() đúng logic", False))

anomaly_path = f"{DATA_OUTPUT}/anomalies"
checks.append(("File Parquet anomalies tồn tại",
    os.path.exists(anomaly_path) and
    any(f.endswith(".parquet") for _,_,fs in os.walk(anomaly_path) for f in fs)))

all_passed = True
for name, passed in checks:
    print(f"  [{'✓' if passed else '✗'}] {name}")
    if not passed: all_passed = False
print()
print("✓ Bài 2 hoàn thành!" if all_passed else "✗ Xem lại phần chưa qua.")